# Analisis Tren Harga Saham Bank Besar di Indonesia (2014–2024)

**Midterm Project – CAK4XBB3 Visualisasi Data**

**Kelompok:** [nomor kelompok]

| No | Nama | NIM |
|---|---|---|
| 1 | Aprilianza Muhammad Yusup | 103012300025 |
| 2 | Arfi | [NIM] |
| 3 | Jeany | [NIM] |
| 4 | Irgi | [NIM] |
| 5 | Bill | [NIM] |

## Daftar Isi

1. Pendahuluan
2. Dataset
3. Visualisasi Time Series
4. Analisis Tren
5. Kesimpulan
6. Deklarasi Penggunaan AI
7. Daftar Pustaka

## 1. Pendahuluan

### 1.1 Latar Belakang

### 1.2 Pertanyaan Penelitian

## 2. Dataset

### 2.1 Sumber Data

Dataset yang digunakan adalah *Dataset Saham Bank Indonesia 2014–2024* dari Kaggle (https://www.kaggle.com/datasets/cmglonly/dataset-saham-bank-indonesia-2014-2024/data). Dataset terdiri dari empat file CSV (satu per bank) yang bersumber dari Yahoo Finance: BBCA, BBRI, BMRI, dan BBNI.

### 2.2 Import Library dan Load Data

Keempat file digabung menjadi satu DataFrame `df_all`, kemudian dibuat tabel `adj_wide` (Adj Close) dan `vol_wide` (Volume) dengan tanggal sebagai index dan bank sebagai kolom.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

TICKERS = ['BBCA', 'BBRI', 'BMRI', 'BBNI']
NAMA_BANK = {
    'BBCA': 'Bank Central Asia',
    'BBRI': 'Bank Rakyat Indonesia',
    'BMRI': 'Bank Mandiri',
    'BBNI': 'Bank Negara Indonesia',
}
WARNA = {'BBCA': '#2a78d6', 'BBRI': '#eb6834', 'BMRI': '#1baf7a', 'BBNI': '#eda100'}

# lokasi folder dataset (root repo / folder kerja), kalau tidak ada ambil dari GitHub
if Path('dataset').is_dir():
    ROOT = Path('.')
elif Path('../dataset').is_dir():
    ROOT = Path('..')
else:
    ROOT = None
DATA_DIR = f'{ROOT}/dataset' if ROOT else 'https://raw.githubusercontent.com/aprilianza/Midterm_DataVisualization/main/dataset'
FIG_DIR = (ROOT or Path('.')) / 'figures'
FIG_DIR.mkdir(exist_ok=True)

list_df = []
for t in TICKERS:
    df = pd.read_csv(f'{DATA_DIR}/{t}.JK_10_years.csv')
    df['Date'] = pd.to_datetime(df['Date'], utc=True).dt.tz_localize(None)
    df['Ticker'] = t
    list_df.append(df.sort_values('Date'))

df_all = pd.concat(list_df, ignore_index=True)
df_all = df_all[['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']]
df_all = df_all.drop_duplicates(['Ticker', 'Date'])

# baris volume 0 dengan harga datar = hari libur bursa, dibuang untuk semua bank
libur = ((df_all['Volume'] == 0) & (df_all['Open'] == df_all['Close'])
         & (df_all['High'] == df_all['Close']) & (df_all['Low'] == df_all['Close']))
HARI_DIBUANG = pd.DatetimeIndex(df_all.loc[libur, 'Date'].unique()).sort_values()
df_all = df_all[~df_all['Date'].isin(HARI_DIBUANG)].reset_index(drop=True)

adj_wide = df_all.pivot(index='Date', columns='Ticker', values='Adj Close')[TICKERS]
vol_wide = df_all.pivot(index='Date', columns='Ticker', values='Volume')[TICKERS]

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'savefig.dpi': 200,
    'savefig.bbox': 'tight',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.titlelocation': 'left',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.color': '#e6e5e0',
    'legend.frameon': False,
})


def simpan_fig(fig, nama):
    fig.savefig(FIG_DIR / f'{nama}.png')


def format_tanggal(ax):
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))


print('Jumlah baris df_all :', len(df_all))
print('Rentang tanggal     :', df_all['Date'].min().date(), 's.d.', df_all['Date'].max().date())
print('Hari libur dibuang  :', len(HARI_DIBUANG))

### 2.3 Profil Dataset

#### Jumlah Observasi dan Rentang Waktu

In [ ]:
ringkas = df_all.groupby('Ticker', sort=False).agg(
    Observasi=('Date', 'size'),
    Mulai=('Date', 'min'),
    Selesai=('Date', 'max'),
)
ringkas.insert(0, 'Nama', ringkas.index.map(NAMA_BANK))
ringkas

#### Missing Value dan Duplikat

Pada data mentah terdapat baris dengan Volume 0 dan harga Open, High, Low, Close yang sama. Tanggal baris tersebut bertepatan dengan hari libur bursa seperti 17 Agustus, Natal, Tahun Baru, dan cuti bersama Lebaran, sehingga bukan hari perdagangan. Baris ini sudah dibuang saat load data (disimpan di `HARI_DIBUANG`).

In [ ]:
print(len(HARI_DIBUANG), 'tanggal dibuang, contoh:')
print(HARI_DIBUANG[:10].strftime('%d %b %Y (%a)').tolist())

In [ ]:
print('Missing value per kolom:')
print(df_all.isna().sum(), end='\n\n')
print('Duplikat (Ticker, Date):', df_all.duplicated(['Ticker', 'Date']).sum())
print('Tanggal yang tidak lengkap di 4 bank:', adj_wide.isna().any(axis=1).sum())
print('Sisa hari dengan Volume = 0:')
print((vol_wide == 0).sum())

#### Granularitas Waktu

Data bersifat harian, tetapi hanya mencakup hari bursa (tanpa Sabtu, Minggu, dan hari libur). Tahun 2014 dan 2024 tidak penuh satu tahun.

In [ ]:
hari_bursa = adj_wide.groupby(adj_wide.index.year).size().rename('Hari bursa')
hari_bursa.to_frame().T

In [ ]:
tgl = adj_wide.index.to_series()
selisih = tgl.diff().dt.days
print('Jarak antar observasi (hari):')
print(selisih.value_counts().sort_index().head(10), end='\n\n')
print('5 jeda terpanjang:')
pd.DataFrame({'Dari': tgl.shift(), 'Sampai': tgl, 'Hari': selisih}).nlargest(5, 'Hari').reset_index(drop=True)

#### Statistik Deskriptif

In [ ]:
adj_wide.describe().round(0)

In [ ]:
# volume dalam juta lembar
(vol_wide.describe().drop('count') / 1e6).round(1)

#### Close vs Adj Close

Pada data Yahoo Finance, kolom `Close` sudah disesuaikan dengan stock split, sedangkan `Adj Close` disesuaikan dengan stock split dan dividen. Karena penyesuaian split, kolom `Close` memiliki harga pecahan (misalnya 2.987,5) pada periode sebelum split, padahal harga saham di BEI selalu bilangan bulat. Untuk membandingkan pertumbuhan harga antar bank, analisis selanjutnya menggunakan `Adj Close`.

In [ ]:
rasio = (df_all['Adj Close'] / df_all['Close']).groupby(df_all['Ticker'], sort=False)
pd.DataFrame({'Rasio awal': rasio.first(), 'Rasio akhir': rasio.last()}).round(3)

In [ ]:
pecahan = df_all[df_all['Close'] % 1 != 0]
pecahan.groupby('Ticker', sort=False)['Date'].agg(['size', 'max']).rename(
    columns={'size': 'Hari Close pecahan', 'max': 'Terakhir muncul'})

## 3. Visualisasi Time Series

### 3.1 Line Plot

### 3.2 Area Plot

### 3.3 Multiple Time Series

### 3.4 Seasonal Plot / Calendar Heatmap

## 4. Analisis Tren

### 4.1 Moving Average

### 4.2 Fitting Model

## 5. Kesimpulan

### 5.1 Insight Utama

### 5.2 Kendala

### 5.3 Pengembangan Selanjutnya

## 6. Deklarasi Penggunaan AI

"Saya, [Nama Lengkap] / [NIM], menyatakan bahwa saya [menggunakan / tidak menggunakan] alat AI generatif dalam pengerjaan tugas ini. Seluruh interpretasi, insight, dan kesimpulan yang disampaikan merupakan hasil pemahaman saya sendiri."

| No | Nama Tools AI | Bagian yang Dibantu | Bentuk Bantuan / Tujuan Prompt | Tingkat Ketergantungan |
|---|---|---|---|---|
| 1 | | | | |

Screenshot prompt dikumpulkan terpisah dalam berkas `Screenshot_AI_[NIM]`.

## 7. Daftar Pustaka

1. Dataset Saham Bank Indonesia 2014–2024, Kaggle. https://www.kaggle.com/datasets/cmglonly/dataset-saham-bank-indonesia-2014-2024/data
2. Yahoo Finance. https://finance.yahoo.com